# PhoBERT fine-tune trên Colab — 5 fold, 8 epoch

**Cách dùng:** viết/sửa notebook trong VSCode trên Windows, chọn **Colab kernel** của extension Google Colab. Toàn bộ cell Python chạy trên GPU của Colab (T4 16 GB), không dùng GPU laptop.

**Cấu hình:** 5 fold, 8 epoch, batch 16, AMP fp16, early stopping (patience 2), class weight `sqrt`.

**Dữ liệu cần có trên Google Drive** (chỉ ~8 MB):

```
MyDrive/ds102/artifacts/
├── phobert/
│   ├── train.parquet
│   ├── dev.parquet
│   └── test.parquet
├── dense/
│   ├── dense_train.npy
│   ├── dense_dev.npy
│   └── dense_test.npy
└── labels/
    ├── y_train.npy
    └── y_dev.npy
```

**Kết quả:** lưu vào `MyDrive/ds102/artifacts/probs/oof_phobert/`, gồm checkpoint từng fold và 3 file cuối:

```
phobert_fold{k}_va.npy / _dev.npy / _test.npy
phobert_oof.npy   (14,306 × 4)
phobert_dev.npy   (1,590 × 4)
phobert_test.npy  (3,974 × 4)
```

Nếu session bị ngắt, chạy lại notebook: các fold đã xong sẽ được **bỏ qua** (`SKIP_EXISTING=True`).

**Ước tính thời gian trên T4:** 5 fold × tối đa 8 epoch, có early stopping → khoảng **1.5–2.5 giờ**.

In [3]:
# ============================================================
# 1. MÔI TRƯỜNG & ĐƯỜNG DẪN
# ============================================================
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("IN_COLAB:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/ds102")
else:
    def find_project_root(start=None, marker="Data/ds102_train.csv"):
        p = Path(start).resolve() if start is not None else Path.cwd().resolve()
        for cand in [p, *p.parents]:
            if (cand / marker).exists():
                return cand
        raise FileNotFoundError(f"Không tìm thấy thư mục gốc chứa '{marker}'")
    PROJECT_ROOT = find_project_root()

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
PHOBERT_TXT_DIR = ARTIFACT_DIR / "phobert"
OOF_DIR = ARTIFACT_DIR / "probs" / "oof_phobert"
OOF_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OOF_DIR     :", OOF_DIR)

required = [
    PHOBERT_TXT_DIR / "train.parquet",
    PHOBERT_TXT_DIR / "dev.parquet",
    PHOBERT_TXT_DIR / "test.parquet",
    ARTIFACT_DIR / "dense" / "dense_train.npy",
    ARTIFACT_DIR / "dense" / "dense_dev.npy",
    ARTIFACT_DIR / "dense" / "dense_test.npy",
    ARTIFACT_DIR / "labels" / "y_train.npy",
    ARTIFACT_DIR / "labels" / "y_dev.npy",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Thiếu file trên Drive:\n  " + "\n  ".join(missing) +
        "\nHãy upload đúng cấu trúc MyDrive/ds102/artifacts/ như mô tả ở đầu notebook."
    )
print("Đủ file đầu vào.")

IN_COLAB: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/ds102
OOF_DIR     : /content/drive/MyDrive/ds102/artifacts/probs/oof_phobert
Đủ file đầu vào.


In [4]:
# ============================================================
# 2. IMPORT & CẤU HÌNH
# ============================================================
import math
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
    from torch.optim import AdamW
    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
except ImportError as exc:
    raise ImportError(
        "Thiếu torch/transformers. Trên Colab thường đã có sẵn; nếu không:\n"
        "  !pip install transformers sentencepiece"
    ) from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("CẢNH BÁO: không có CUDA — train trên CPU sẽ rất chậm.")

MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 128
BATCH_SIZE = 16 if IN_COLAB else 8
EPOCHS = 8
PATIENCE = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS = 5
SEED = 42
CLASS_WEIGHT_MODE = "sqrt"     # sqrt | balanced | none
GRAD_CHECKPOINT = not IN_COLAB
NUM_WORKERS = 2 if IN_COLAB else 0
SKIP_EXISTING = True           # bỏ qua fold đã có checkpoint

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"config: folds={N_FOLDS} epochs={EPOCHS} batch={BATCH_SIZE} "
      f"max_len={MAX_LEN} patience={PATIENCE} amp={USE_AMP}")

torch: 2.11.0+cu130 | CUDA: 13.0
GPU: Tesla T4 | VRAM: 15.6 GB
config: folds=5 epochs=8 batch=16 max_len=128 patience=2 amp=True


In [5]:
# ============================================================
# 3. NẠP DỮ LIỆU
# ============================================================
train_df = pd.read_parquet(PHOBERT_TXT_DIR / "train.parquet")
dev_df = pd.read_parquet(PHOBERT_TXT_DIR / "dev.parquet")
test_df = pd.read_parquet(PHOBERT_TXT_DIR / "test.parquet")

texts_train = train_df["comment_seg"].fillna("").tolist()
texts_dev = dev_df["comment_seg"].fillna("").tolist()
texts_test = test_df["comment_seg"].fillna("").tolist()

y_train = np.load(ARTIFACT_DIR / "labels" / "y_train.npy")
y_dev = np.load(ARTIFACT_DIR / "labels" / "y_dev.npy")

dense_train = np.load(ARTIFACT_DIR / "dense" / "dense_train.npy").astype(np.float32)
dense_dev = np.load(ARTIFACT_DIR / "dense" / "dense_dev.npy").astype(np.float32)
dense_test = np.load(ARTIFACT_DIR / "dense" / "dense_test.npy").astype(np.float32)

assert len(texts_train) == len(y_train) == len(dense_train)
assert len(texts_dev) == len(y_dev) == len(dense_dev)
assert len(texts_test) == len(dense_test)

print("train:", len(texts_train), "| dev:", len(texts_dev), "| test:", len(texts_test))
print("dense dims:", dense_train.shape[1])
print("nhãn train:", dict(zip(*np.unique(y_train, return_counts=True))))

train: 14306 | dev: 1590 | test: 3974
dense dims: 52
nhãn train: {np.int64(0): np.int64(10512), np.int64(1): np.int64(211), np.int64(2): np.int64(1103), np.int64(3): np.int64(2480)}


In [6]:
# ============================================================
# 4. TOKENIZER & DATASET
# ============================================================
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    print("Dùng fast tokenizer.")
except Exception as exc:
    print("Fast tokenizer lỗi:", exc)
    print("Thử slow tokenizer (có thể cần: pip install fastBPE).")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def encode(texts):
    return tokenizer(
        texts, truncation=True, max_length=MAX_LEN,
        padding="max_length", return_tensors="pt",
    )


print("Đang tokenize...")
enc_train = encode(texts_train)
enc_dev = encode(texts_dev)
enc_test = encode(texts_test)
print("enc_train:", enc_train["input_ids"].shape)


def subset_enc(enc, idx):
    idx = torch.as_tensor(idx, dtype=torch.long)
    return {
        "input_ids": enc["input_ids"][idx],
        "attention_mask": enc["attention_mask"][idx],
    }


class ReviewDataset(Dataset):
    def __init__(self, enc, dense, labels=None):
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.dense = torch.tensor(dense, dtype=torch.float32)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.input_ids.shape[0]

    def __getitem__(self, i):
        item = {
            "input_ids": self.input_ids[i],
            "attention_mask": self.attention_mask[i],
            "dense": self.dense[i],
        }
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


print("Đã tạo Dataset.")

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Dùng fast tokenizer.
Đang tokenize...
enc_train: torch.Size([14306, 128])
Đã tạo Dataset.


In [7]:
# ============================================================
# 5. MODEL
# ============================================================
class PhoBERTMeta(nn.Module):
    def __init__(self, model_name, n_dense, n_classes=4, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dense_proj = nn.Sequential(
            nn.Linear(n_dense, 64), nn.ReLU(), nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 64, n_classes)

    def forward(self, input_ids, attention_mask, dense):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0]
        d = self.dense_proj(dense)
        return self.classifier(self.dropout(torch.cat([cls, d], dim=1)))


def class_weights(y, mode):
    counts = np.bincount(y, minlength=4).astype(float)
    balanced = len(y) / (4.0 * np.maximum(counts, 1.0))
    if mode == "balanced":
        vals = balanced
    elif mode == "sqrt":
        vals = np.sqrt(balanced)
    else:
        vals = np.ones(4)
    return torch.tensor(vals, dtype=torch.float32)


print("Weight sqrt:", class_weights(y_train, "sqrt").numpy().round(2).tolist())

Weight sqrt: [0.5799999833106995, 4.119999885559082, 1.7999999523162842, 1.2000000476837158]


In [8]:
# ============================================================
# 6. TRAIN MỘT FOLD
# ============================================================
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        dense = batch["dense"].to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = model(input_ids, attention_mask, dense)
        probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(probs)


def run_fold(fold, tr_idx, va_idx):
    t0 = time.time()

    scaler = StandardScaler().fit(dense_train[tr_idx])
    d_tr = scaler.transform(dense_train[tr_idx]).astype(np.float32)
    d_va = scaler.transform(dense_train[va_idx]).astype(np.float32)
    d_dv = scaler.transform(dense_dev).astype(np.float32)
    d_te = scaler.transform(dense_test).astype(np.float32)

    train_ds = ReviewDataset(subset_enc(enc_train, tr_idx), d_tr, y_train[tr_idx])
    va_ds = ReviewDataset(subset_enc(enc_train, va_idx), d_va)
    dev_ds = ReviewDataset(enc_dev, d_dv)
    test_ds = ReviewDataset(enc_test, d_te)

    pin = DEVICE == "cuda"
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=pin)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=pin)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=pin)

    model = PhoBERTMeta(MODEL_NAME, dense_train.shape[1]).to(DEVICE)
    if GRAD_CHECKPOINT and hasattr(model.encoder, "gradient_checkpointing_enable"):
        model.encoder.gradient_checkpointing_enable()

    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = EPOCHS * math.ceil(len(train_ds) / BATCH_SIZE)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )
    loss_fn = nn.CrossEntropyLoss(
        weight=class_weights(y_train[tr_idx], CLASS_WEIGHT_MODE).to(DEVICE)
    )
    amp_scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    best_f1 = -1.0
    best_state = None
    stale = 0

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            dense = batch["dense"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
                logits = model(input_ids, attention_mask, dense)
                loss = loss_fn(logits, labels)

            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            amp_scaler.step(optimizer)
            amp_scaler.update()
            scheduler.step()
            epoch_loss += float(loss.item())

        p_dev = predict(model, dev_loader)
        dev_f1 = f1_score(y_dev, p_dev.argmax(axis=1), average="macro")
        dev_acc = accuracy_score(y_dev, p_dev.argmax(axis=1))
        print(f"  fold {fold} | epoch {epoch + 1}/{EPOCHS} | "
              f"loss={epoch_loss / max(len(train_loader), 1):.4f} | "
              f"dev mf1={dev_f1:.4f} acc={dev_acc:.4f}", flush=True)

        if dev_f1 > best_f1 + 1e-6:
            best_f1 = dev_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"  fold {fold}: early stopping ở epoch {epoch + 1}", flush=True)
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    p_va = predict(model, va_loader)
    p_dev = predict(model, dev_loader)
    p_test = predict(model, test_loader)

    va_f1 = f1_score(y_train[va_idx], p_va.argmax(axis=1), average="macro")
    print(f"  fold {fold} xong | val mf1={va_f1:.4f} | best dev mf1={best_f1:.4f} | "
          f"{(time.time() - t0) / 60:.1f} phút", flush=True)

    del model, optimizer, scheduler, train_loader, va_loader, dev_loader, test_loader
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return p_va, p_dev, p_test, best_f1

## 7. Chạy 5 fold

Mỗi fold lưu ngay 3 file `phobert_fold{k}_{va,dev,test}.npy` vào Drive. Nếu session bị ngắt, chạy lại cell này — các fold đã xong sẽ được bỏ qua.

In [9]:
# ============================================================
# 7. OOF LOOP 5 FOLD
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(np.zeros(len(y_train)), y_train))

oof = np.zeros((len(y_train), 4), dtype=np.float32)
dev_sum = np.zeros((len(y_dev), 4), dtype=np.float32)
test_sum = np.zeros((len(texts_test), 4), dtype=np.float32)

t_all = time.time()
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    print(f"\n===== FOLD {fold} =====", flush=True)
    p_va_path = OOF_DIR / f"phobert_fold{fold}_va.npy"
    p_dev_path = OOF_DIR / f"phobert_fold{fold}_dev.npy"
    p_test_path = OOF_DIR / f"phobert_fold{fold}_test.npy"

    if SKIP_EXISTING and p_va_path.exists() and p_dev_path.exists() and p_test_path.exists():
        p_va = np.load(p_va_path)
        p_dev = np.load(p_dev_path)
        p_test = np.load(p_test_path)
        va_f1 = f1_score(y_train[va_idx], p_va.argmax(axis=1), average="macro")
        print(f"  fold {fold} đã có checkpoint — bỏ qua | val mf1={va_f1:.4f}", flush=True)
        fold_scores.append(va_f1)
    else:
        p_va, p_dev, p_test, best_dev_f1 = run_fold(fold, tr_idx, va_idx)
        np.save(p_va_path, p_va)
        np.save(p_dev_path, p_dev)
        np.save(p_test_path, p_test)
        va_f1 = f1_score(y_train[va_idx], p_va.argmax(axis=1), average="macro")
        fold_scores.append(va_f1)
        print(f"  đã lưu checkpoint fold {fold}", flush=True)

    oof[va_idx] = p_va
    dev_sum += p_dev / N_FOLDS
    test_sum += p_test / N_FOLDS

oof_pred = oof.argmax(axis=1)
print(f"\nOOF macro-F1 = {f1_score(y_train, oof_pred, average='macro'):.4f} | "
      f"accuracy = {accuracy_score(y_train, oof_pred):.4f}")
print(f"Tổng thời gian: {(time.time() - t_all) / 60:.1f} phút")


===== FOLD 0 =====


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

... (10 outputs left out)

  fold 3 | epoch 1/8 | loss=0.7796 | dev mf1=0.6442 acc=0.8384
  fold 3 | epoch 2/8 | loss=0.4342 | dev mf1=0.6770 acc=0.8830
  fold 3 | epoch 3/8 | loss=0.3331 | dev mf1=0.6882 acc=0.8962
  fold 3 | epoch 4/8 | loss=0.2623 | dev mf1=0.6874 acc=0.8969
  fold 3 | epoch 5/8 | loss=0.1842 | dev mf1=0.7020 acc=0.8925
  fold 3 | epoch 6/8 | loss=0.1339 | dev mf1=0.6836 acc=0.8899
  fold 3 | epoch 7/8 | loss=0.0944 | dev mf1=0.6769 acc=0.8874
  fold 3: early stopping ở epoch 7
  fold 3 xong | val mf1=0.7559 | best dev mf1=0.7020 | 12.9 phút
  đã lưu checkpoint fold 3

===== FOLD 4 =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  fold 4 | epoch 1/8 | loss=0.7637 | dev mf1=0.6610 acc=0.8667
  fold 4 | epoch 2/8 | loss=0.4311 | dev mf1=0.7111 acc=0.8843
  fold 4 | epoch 3/8 | loss=0.3136 | dev mf1=0.7010 acc=0.9013
  fold 4 | epoch 4/8 | loss=0.2329 | dev mf1=0.6955 acc=0.9019
  fold 4: early stopping ở epoch 4
  fold 4 xong | val mf1=0.7404 | best dev mf1=0.7111 | 7.5 phút
  đã lưu checkpoint fold 4

OOF macro-F1 = 0.7509 | accuracy = 0.8967
Tổng thời gian: 53.6 phút


In [10]:
# ============================================================
# 8. LƯU KẾT QUẢ CUỐI
# ============================================================
np.save(OOF_DIR / "phobert_oof.npy", oof)
np.save(OOF_DIR / "phobert_dev.npy", dev_sum)
np.save(OOF_DIR / "phobert_test.npy", test_sum)

print("Đã lưu:")
for name in ["phobert_oof.npy", "phobert_dev.npy", "phobert_test.npy"]:
    p = OOF_DIR / name
    print(f"  {p}  ({p.stat().st_size / 1e6:.2f} MB)")

if IN_COLAB:
    print("\nTrên Colab, tải 3 file về máy:")
    print("  from google.colab import files")
    print("  files.download(str(OOF_DIR / 'phobert_oof.npy'))")
    print("  files.download(str(OOF_DIR / 'phobert_dev.npy'))")
    print("  files.download(str(OOF_DIR / 'phobert_test.npy'))")
    print("Hoặc lấy trực tiếp từ Drive: MyDrive/ds102/artifacts/probs/oof_phobert/")
else:
    print("\nCopy 3 file này vào artifacts/probs/oof_phobert/ trên máy local rồi chạy stacking.")

Đã lưu:
  /content/drive/MyDrive/ds102/artifacts/probs/oof_phobert/phobert_oof.npy  (0.23 MB)
  /content/drive/MyDrive/ds102/artifacts/probs/oof_phobert/phobert_dev.npy  (0.03 MB)
  /content/drive/MyDrive/ds102/artifacts/probs/oof_phobert/phobert_test.npy  (0.06 MB)

Trên Colab, tải 3 file về máy:
  from google.colab import files
  files.download(str(OOF_DIR / 'phobert_oof.npy'))
  files.download(str(OOF_DIR / 'phobert_dev.npy'))
  files.download(str(OOF_DIR / 'phobert_test.npy'))
Hoặc lấy trực tiếp từ Drive: MyDrive/ds102/artifacts/probs/oof_phobert/


## 9. Ghi chú

- **Chống ngắt session:** mỗi fold lưu checkpoint riêng; chạy lại sẽ bỏ qua fold đã xong.
- **8 epoch + patience 2:** early stopping sẽ dừng nếu dev macro-F1 không tăng.
- **Early stopping dùng dev:** dev có thể lạc quan nhẹ (dev tham gia chọn epoch). Nếu cần sạch tuyệt đối, có thể tách 10% fold-train làm validation nội bộ.
- **Sau khi xong:** tải 3 file `phobert_{oof,dev,test}.npy` về local, đặt vào `artifacts/probs/oof_phobert/`, rồi chạy `src/stacking.ipynb`.
- **Nếu OOM:** giảm `BATCH_SIZE = 8`, bật `GRAD_CHECKPOINT = True`.
- **Nếu tokenizer lỗi:** `pip install fastBPE` hoặc dùng `use_fast=False`.